# UAVid++ ViT-H+ + UNet — Single Image Inference (clean rebuild)

This notebook is a cleaned-up, working version of the UAVid++ inference pipeline.

**What was broken and what this fixes:**
- The official DINOv3 repo (`facebookresearch/dinov3`) was never actually cloned into `models/dinov3` in a reproducible cell — later cells assumed it already existed.
- `DinoV3SemanticSegmentationRegisters(weights_name=None)` crashed, because the wrapper always calls `torch.hub.load(..., weights=None)` with `pretrained` defaulting to `True`. DINOv3 then tries to build a download URL from `None` and throws `TypeError: argument should be a str or an os.PathLike object ... not 'NoneType'`.
- You don't actually need the separate (license-gated) DINOv3 ImageNet backbone checkpoint at all: `best_model.pth` already contains the **fully trained backbone + head** (620 tensors: 552 backbone + 68 classifier head). So we build the backbone architecture with random init (`pretrained=False`) and then load your trained checkpoint on top of it, which fills in every weight.
- All of the exploratory/debugging cells (checkpoint key dumps, repeated inspections, dead imports like `DINOv3Wrapper`) have been removed. What's left is the minimal path from a fresh runtime to a segmented image.

**How to use:** Run the cells top to bottom. You'll be asked to (1) paste a Hugging Face token to download the trained checkpoint, and (2) manually upload your test image when prompted.

## 1. Clone the UAVid++ repository

In [1]:
!git clone https://github.com/vivichiciudean/uavidplusplus-code.git
%cd /content/uavidplusplus-code
!ls -lah


Cloning into 'uavidplusplus-code'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 57 (delta 7), reused 45 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 82.64 KiB | 1.42 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/uavidplusplus-code
total 52K
drwxr-xr-x 8 root root 4.0K Sep 17 08:50 .
drwxr-xr-x 1 root root 4.0K Sep 17 08:50 ..
drwxr-xr-x 3 root root 4.0K Sep 17 08:50 code
drwxr-xr-x 2 root root 4.0K Sep 17 08:50 data
drwxr-xr-x 2 root root 4.0K Sep 17 08:50 data_processing
-rw-r--r-- 1 root root  337 Sep 17 08:50 env.yml
drwxr-xr-x 8 root root 4.0K Sep 17 08:50 .git
drwxr-xr-x 2 root root 4.0K Sep 17 08:50 models
drwxr-xr-x 2 root root 4.0K Sep 17 08:50 output
-rw-r--r-- 1 root root  11K Sep 17 08:50 README.md
-rw-r--r-- 1 root root  132 Sep 17 08:50 req.txt


## 2. Clone the official DINOv3 repository into `models/`

The UAVid++ code imports the DINOv3 architecture via `torch.hub.load(..., source='local')`, which requires the official DINOv3 repo to be physically present at `models/dinov3`.

In [2]:
import os

DINO_REPO = "/content/uavidplusplus-code/models/dinov3"

if not os.path.exists(os.path.join(DINO_REPO, "hubconf.py")):
    !git clone https://github.com/facebookresearch/dinov3.git {DINO_REPO}
else:
    print("DINOv3 repo already present, skipping clone.")

print("hubconf.py exists:", os.path.exists(os.path.join(DINO_REPO, "hubconf.py")))


Cloning into '/content/uavidplusplus-code/models/dinov3'...
remote: Enumerating objects: 660, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 660 (delta 195), reused 154 (delta 154), pack-reused 332 (from 1)
Receiving objects: 100% (660/660), 12.99 MiB | 24.86 MiB/s, done.
Resolving deltas: 100% (269/269), done.
hubconf.py exists: True


## 3. Install dependencies

In [3]:
!pip -q install \
    torch \
    torchvision \
    timm \
    opencv-python \
    pillow \
    numpy \
    matplotlib \
    tqdm \
    einops \
    transformers \
    huggingface_hub


## 4. Check GPU

In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("WARNING: GPU not available. Go to Runtime > Change runtime type > T4 GPU.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


## 5. Hugging Face login + download the trained checkpoint

In [5]:
from huggingface_hub import login
from getpass import getpass

token = getpass("Enter your Hugging Face access token: ")
login(token=token)


Enter your Hugging Face access token: ··········


In [6]:
from huggingface_hub import hf_hub_download
import os

REPO_ID = "vivianchiciudean/uavidplusplus"
FILENAME = "output_vith+_unet_uavid++.zip"

MODEL_ZIP = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, repo_type="dataset")

print("Downloaded:", MODEL_ZIP)
print("Size:", round(os.path.getsize(MODEL_ZIP) / (1024**3), 2), "GB")


output_vith+_unet_uavid++.zip: reconstructing file:   0%|          |  0.00B / 3.42GB            

output_vith+_unet_uavid++.zip: downloading bytes:           |  0.00B            

Downloaded: /root/.cache/huggingface/hub/datasets--vivianchiciudean--uavidplusplus/snapshots/d7365c031bad573754d78dc08e1914620e4720ec/output_vith+_unet_uavid++.zip
Size: 3.19 GB


## 6. Extract the checkpoint

In [7]:
import zipfile
import os

MODEL_DIR = "/content/uavid_vith_model"
os.makedirs(MODEL_DIR, exist_ok=True)

with zipfile.ZipFile(MODEL_ZIP, "r") as z:
    z.extractall(MODEL_DIR)

CHECKPOINT_PATH = None
for root, dirs, files in os.walk(MODEL_DIR):
    for f in files:
        if f.endswith(".pth"):
            CHECKPOINT_PATH = os.path.join(root, f)

assert CHECKPOINT_PATH is not None, "No .pth checkpoint found after extraction."

print("Checkpoint:", CHECKPOINT_PATH)
print("Size:", round(os.path.getsize(CHECKPOINT_PATH) / (1024**3), 2), "GB")


Checkpoint: /content/uavid_vith_model/output_vith+_unet_uavid++/best_model.pth
Size: 3.42 GB


## 7. Patch `dinov3_wrapper.py`

The wrapper always calls `torch.hub.load(..., weights=self.weights_name)` and leaves `pretrained` at its default (`True`). When `weights_name=None` (our case — we don't need the separate DINOv3 backbone checkpoint), DINOv3 tries to convert `None` into a download URL and crashes.

This patch makes the wrapper pass `pretrained=False` whenever `weights_name` is `None`, so it builds the architecture with random initial weights instead of trying to download anything — those random weights get fully overwritten in step 9 by your trained `best_model.pth`. If you do pass a real `weights_name` path, behavior is unchanged.

In [8]:
wrapper_path = "/content/uavidplusplus-code/code/dinov3_wrapper.py"

with open(wrapper_path, "r", encoding="utf-8") as f:
    src = f.read()

old_line = "self.backbone = torch.hub.load(repo_or_dir=self.repo_name, model=self.model_name, source='local', weights=self.weights_name)"
new_line = "self.backbone = torch.hub.load(repo_or_dir=self.repo_name, model=self.model_name, source='local', weights=self.weights_name, pretrained=(self.weights_name is not None))"

if old_line in src:
    src = src.replace(old_line, new_line)
    with open(wrapper_path, "w", encoding="utf-8") as f:
        f.write(src)
    print("Patched dinov3_wrapper.py")
elif new_line in src:
    print("Already patched, nothing to do.")
else:
    raise RuntimeError(
        "Could not find the expected torch.hub.load line to patch. "
        "The repo may have been updated upstream — open dinov3_wrapper.py and check the backbone-loading line manually."
    )


Patched dinov3_wrapper.py


## 8. Imports from the official code

In [9]:
import sys

REPO = "/content/uavidplusplus-code"
CODE_DIR = f"{REPO}/code"

for p in (REPO, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

# Reload in case this cell is re-run after the patch above
for mod in ("dinov3_wrapper", "dataloader", "inference_strategy"):
    sys.modules.pop(mod, None)

from dataloader import num_labels, id2color_np, id2name
from dinov3_wrapper import DinoV3SemanticSegmentationRegisters
from inference_strategy import inference_tiled

print("num_labels:", num_labels)
print("classes:", list(id2name.values()))


num_labels: 11
classes: ['clutter', 'wall', 'road', 'tree', 'lowveg', 'water', 'sky', 'roof', 'staticcar', 'dynamiccar', 'human']


## 9. Build the model and load your trained checkpoint

In [10]:
!pip -q install torchmetrics

import torch

REPO_NAME = f"{REPO}/models/dinov3"
MODEL_NAME = "dinov3_vith16plus"
PATCH_SIZE = 16
IGNORE_INDEX = 255

print("Creating DINOv3 ViT-H+ + UNet model (random-init backbone, will be overwritten below)...")

model = DinoV3SemanticSegmentationRegisters(
    num_labels=num_labels,
    repo_name=REPO_NAME,
    model_name=MODEL_NAME,
    half_precision=False,
    device=device,
    class_weights=None,
    weights_name=None,   # we don't need the separate DINOv3 backbone checkpoint
    patch_size=PATCH_SIZE,
    ignore_index=IGNORE_INDEX,
)
model = model.to(device)

print("\nLoading trained checkpoint...")
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state_dict = checkpoint["model_state_dict"]

print("Checkpoint epoch      :", checkpoint.get("epoch", "?"))
print("Checkpoint best val IoU:", checkpoint.get("best_val_iou", "?"))
print("Checkpoint tensors    :", len(state_dict))

try:
    model.load_state_dict(state_dict, strict=True)
    print("\n✅ Loaded checkpoint with strict=True (all keys matched).")
except RuntimeError as e:
    print("\nstrict=True failed, retrying with strict=False. Details:")
    print(e)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Missing keys   :", len(missing))
    print("Unexpected keys:", len(unexpected))

model.eval()
del checkpoint, state_dict
torch.cuda.empty_cache() if device.type == "cuda" else None

print("\n✅ Model ready on", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 24.3 MB/s eta 0:00:00
Creating DINOv3 ViT-H+ + UNet model (random-init backbone, will be overwritten below)...
In channels dim dinov3 (should be 4096 for vit7b)
1280
Trainable parameters: 25397451

Loading trained checkpoint...
Checkpoint epoch      : 34
Checkpoint best val IoU: 0.8126911203090963
Checkpoint tensors    : 620

✅ Loaded checkpoint with strict=True (all keys matched).

✅ Model ready on cuda


## 10. Upload your test image

In [11]:
from google.colab import files
import os

print("Choose an image file to upload...")
uploaded = files.upload()

assert len(uploaded) > 0, "No file uploaded."
TEST_IMAGE = "/content/" + list(uploaded.keys())[-1]
print("\nUsing image:", TEST_IMAGE)


Choose an image file to upload...


AssertionError: No file uploaded.

## 11. Run inference and visualize the result

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

original = Image.open(TEST_IMAGE).convert("RGB")
original_np = np.asarray(original, dtype=np.uint8)
orig_w, orig_h = original.size

print("Original image:", orig_w, "x", orig_h)

# ------------------------------------------------------------
# Normalize (same stats used by the official UAVid++ inference)
# ------------------------------------------------------------
imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

image_float = original_np.astype(np.float32) / 255.0
image_normalized = (image_float - imagenet_mean) / imagenet_std

image_tensor = torch.from_numpy(image_normalized).permute(2, 0, 1).float().to(device)

# ------------------------------------------------------------
# Tiled inference: official config is 1088x1088 tiles, 64px overlap
# ------------------------------------------------------------
print("\nRunning tiled inference (1088x1088 tiles, 64px overlap)...")

with torch.inference_mode():
    logits = inference_tiled(
        model,
        image_tensor,
        tile_size=(1088, 1088),
        overlap=64,
        num_labels=num_labels,
        device=device,
    )

prediction = logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
color_mask = id2color_np[prediction]

if color_mask.shape[:2] != original_np.shape[:2]:
    color_mask = np.asarray(
        Image.fromarray(color_mask).resize((orig_w, orig_h), Image.Resampling.NEAREST)
    )

overlay = np.clip(0.55 * original_np.astype(np.float32) + 0.45 * color_mask.astype(np.float32), 0, 255).astype(np.uint8)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
OUTPUT_DIR = "/content/uavid_single_image_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Image.fromarray(original_np).save(os.path.join(OUTPUT_DIR, "real_image.png"))
Image.fromarray(color_mask).save(os.path.join(OUTPUT_DIR, "segmentation_mask.png"))
Image.fromarray(overlay).save(os.path.join(OUTPUT_DIR, "overlay.png"))

# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("PREDICTED CLASSES")
print("=" * 70)

unique, counts = np.unique(prediction, return_counts=True)
total_pixels = prediction.size

for cls_id, count in zip(unique, counts):
    name = id2name.get(int(cls_id), f"class_{cls_id}")
    pct = 100.0 * count / total_pixels
    print(f"{cls_id:2d} | {name:12s} | {count:10d} px | {pct:6.2f}%")

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(original_np)
plt.title("REAL IMAGE", fontsize=14)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(color_mask)
plt.title("UAVid++ MODEL OUTPUT", fontsize=14)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title("REAL + SEGMENTATION OVERLAY", fontsize=14)
plt.axis("off")

plt.tight_layout()
plt.show()

print("\nSaved outputs to:", OUTPUT_DIR)
print("\n✅ INFERENCE COMPLETE")


In [14]:
# ============================================================
# RUN INFERENCE ON A VIDEO AND PLAY IT INLINE
# ============================================================
import cv2
import numpy as np
import torch
from tqdm.auto import tqdm
import os, shutil, glob, json, subprocess

VIDEO_PATH = "/content/temp.mp4"
assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"

OUTPUT_DIR = "/content/uavid_video_output"
IN_FRAMES_DIR = os.path.join(OUTPUT_DIR, "in_frames")
OUT_FRAMES_DIR = os.path.join(OUTPUT_DIR, "out_frames")
for d in (IN_FRAMES_DIR, OUT_FRAMES_DIR):
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
FINAL_OUTPUT = os.path.join(OUTPUT_DIR, "overlay.mp4")

# ------------------------------------------------------------
# Read real video metadata with ffprobe (cv2's metadata was unreliable
# here since cv2 couldn't decode this codec at all)
# ------------------------------------------------------------
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0",
     "-show_entries", "stream=width,height,r_frame_rate",
     "-of", "json", VIDEO_PATH],
    capture_output=True, text=True
)
info = json.loads(probe.stdout)["streams"][0]
width, height = info["width"], info["height"]
num, den = info["r_frame_rate"].split("/")
fps = float(num) / float(den)

print(f"Video: {width}x{height} @ {fps:.2f} fps")

# ------------------------------------------------------------
# Extract frames with ffmpeg (robust across codecs)
# ------------------------------------------------------------
print("\nExtracting frames with ffmpeg...")
!ffmpeg -y -loglevel error -i {VIDEO_PATH} {IN_FRAMES_DIR}/frame_%06d.png

frame_paths = sorted(glob.glob(os.path.join(IN_FRAMES_DIR, "*.png")))
n_frames = len(frame_paths)
print(f"Extracted {n_frames} frames")
assert n_frames > 0, "ffmpeg extracted no frames — check that VIDEO_PATH is a valid video."

imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

model.eval()

with torch.inference_mode():
    for i, frame_path in enumerate(tqdm(frame_paths, desc="Segmenting frames")):
        frame_bgr = cv2.imread(frame_path)
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        image_float = frame_rgb.astype(np.float32) / 255.0
        image_normalized = (image_float - imagenet_mean) / imagenet_std
        image_tensor = torch.from_numpy(image_normalized).permute(2, 0, 1).float().to(device)

        logits = inference_tiled(
            model,
            image_tensor,
            tile_size=(1088, 1088),
            overlap=64,
            num_labels=num_labels,
            device=device,
        )

        prediction = logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
        color_mask = id2color_np[prediction]

        if color_mask.shape[:2] != frame_rgb.shape[:2]:
            color_mask = cv2.resize(color_mask, (frame_rgb.shape[1], frame_rgb.shape[0]), interpolation=cv2.INTER_NEAREST)

        overlay_rgb = np.clip(
            0.55 * frame_rgb.astype(np.float32) + 0.45 * color_mask.astype(np.float32), 0, 255
        ).astype(np.uint8)
        overlay_bgr = cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR)

        cv2.imwrite(os.path.join(OUT_FRAMES_DIR, f"frame_{i:06d}.png"), overlay_bgr)

print("\nEncoding video with ffmpeg...")
!ffmpeg -y -loglevel error -framerate {fps} -i {OUT_FRAMES_DIR}/frame_%06d.png -vcodec libx264 -pix_fmt yuv420p {FINAL_OUTPUT}

assert os.path.exists(FINAL_OUTPUT) and os.path.getsize(FINAL_OUTPUT) > 0, "ffmpeg failed to produce an output file."
print("\n✅ Saved:", FINAL_OUTPUT)

# ------------------------------------------------------------
# Display inline
# ------------------------------------------------------------
from IPython.display import HTML
from base64 import b64encode

video_bytes = open(FINAL_OUTPUT, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(video_bytes).decode()

HTML(f"""
<video width=640 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")

Video: 608x1080 @ 30.00 fps

Extracting frames with ffmpeg...
Extracted 338 frames


Segmenting frames:   0%|          | 0/338 [00:00<?, ?it/s]


Encoding video with ffmpeg...

✅ Saved: /content/uavid_video_output/overlay.mp4
